In [1]:
# Kaggle install karo
!pip install kaggle -q

# Kaggle folder banao
import os
os.makedirs('/root/.kaggle', exist_ok=True)

print("Ready! Ab kaggle.json upload karenge")


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


OSError: [Errno 30] Read-only file system: '/root'

In [ ]:
# Step 1 — sab install karo
!pip install torch torchvision timm transformers -q
print("✅ Libraries ready!")

✅ Libraries ready!


In [ ]:
# PlantVillage dataset download karo
!git clone https://github.com/spMohanty/PlantVillage-Dataset.git
print("✅ Dataset download ho gaya!")

# Kitni images hain check karo
import os
total = 0
for root, dirs, files in os.walk("PlantVillage-Dataset"):
    imgs = [f for f in files if f.lower().endswith(('.jpg','.jpeg','.png'))]
    total += len(imgs)
    if imgs:
        print(f"  {len(imgs)} images — {root.split('/')[-1]}")

print(f"\nTOTAL: {total} images!")

Cloning into 'PlantVillage-Dataset'...
remote: Enumerating objects: 163264, done.
remote: Counting objects: 100% (35/35), done.
remote: Compressing objects: 100% (26/26), done.
remote: Total 163264 (delta 16), reused 25 (delta 9), pack-reused 163229 (from 1)
Receiving objects: 100% (163264/163264), 2.00 GiB | 32.90 MiB/s, done.
Resolving deltas: 100% (115/115), done.
Updating files: 100% (182404/182404), done.
✅ Dataset download ho gaya!
  1676 images — Tomato___Spider_mites Two-spotted_spider_mite
  1383 images — Grape___Esca_(Black_Measles)
  456 images — Strawberry___healthy
  275 images — Apple___Cedar_apple_rust
  854 images — Cherry_(including_sour)___healthy
  1771 images — Tomato___Septoria_leaf_spot
  1000 images — Potato___Late_blight
  360 images — Peach___healthy
  1000 images — Potato___Early_blight
  5357 images — Tomato___Tomato_Yellow_Leaf_Curl_Virus
  513 images — Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot
  1909 images — Tomato___Late_blight
  1645 images — Ap

In [ ]:
import os, shutil
import torch
import torchvision
from torchvision import datasets, transforms, models
from torch import nn, optim
from torch.utils.data import DataLoader

# Rajasthan crops select karo
SELECTED = {
    'Tomato___Early_blight':        'टमाटर_Early_Blight',
    'Tomato___Late_blight':         'टमाटर_Late_Blight',
    'Tomato___healthy':             'टमाटर_Healthy',
    'Potato___Early_blight':        'आलू_Early_Blight',
    'Potato___Late_blight':         'आलू_Late_Blight',
    'Potato___healthy':             'आलू_Healthy',
    'Corn_(maize)___Common_rust_':  'मक्का_Rust',
    'Corn_(maize)___Northern_Leaf_Blight': 'मक्का_Blight',
    'Corn_(maize)___healthy':       'मक्का_Healthy',
    'Pepper,_bell___Bacterial_spot':'मिर्च_Bacterial_Spot',
    'Pepper,_bell___healthy':       'मिर्च_Healthy',
    'Soybean___healthy':            'सोयाबीन_Healthy',
}

# Data folder banao
base = 'PlantVillage-Dataset/raw/color'
out  = 'rajasthan_data'
os.makedirs(out, exist_ok=True)

print("Copying selected images...")
for folder, label in SELECTED.items():
    src = os.path.join(base, folder)
    dst = os.path.join(out, label)
    if os.path.exists(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
        count = len(os.listdir(dst))
        print(f"  ✅ {label}: {count} images")
    else:
        print(f"  ❌ Not found: {folder}")

print("\n✅ Data ready!")

Copying selected images...
  ✅ टमाटर_Early_Blight: 1000 images
  ✅ टमाटर_Late_Blight: 1909 images
  ✅ टमाटर_Healthy: 1591 images
  ✅ आलू_Early_Blight: 1000 images
  ✅ आलू_Late_Blight: 1000 images
  ✅ आलू_Healthy: 152 images
  ✅ मक्का_Rust: 1192 images
  ✅ मक्का_Blight: 985 images
  ✅ मक्का_Healthy: 1162 images
  ✅ मिर्च_Bacterial_Spot: 997 images
  ✅ मिर्च_Healthy: 1478 images
  ✅ सोयाबीन_Healthy: 5090 images

✅ Data ready!


In [ ]:
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
from torch import nn, optim
import torch

# Image transformations
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# Dataset load karo
dataset = datasets.ImageFolder('rajasthan_data', transform=transform)
classes = dataset.classes
print(f"✅ Classes: {len(classes)}")
print(f"✅ Total images: {len(dataset)}")
for i, c in enumerate(classes):
    print(f"   {i}: {c}")

✅ Classes: 12
✅ Total images: 17556
   0: आलू_Early_Blight
   1: आलू_Healthy
   2: आलू_Late_Blight
   3: टमाटर_Early_Blight
   4: टमाटर_Healthy
   5: टमाटर_Late_Blight
   6: मक्का_Blight
   7: मक्का_Healthy
   8: मक्का_Rust
   9: मिर्च_Bacterial_Spot
   10: मिर्च_Healthy
   11: सोयाबीन_Healthy


In [ ]:
# 80% training, 20% testing
train_size = int(0.8 * len(dataset))
test_size  = len(dataset) - train_size

train_data, test_data = random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_data,  batch_size=32, shuffle=False)

print(f"✅ Training images : {train_size}")
print(f"✅ Testing images  : {test_size}")
print(f"✅ Batches in train: {len(train_loader)}")
print("Ready to train! 🚀")

✅ Training images : 14044
✅ Testing images  : 3512
✅ Batches in train: 439
Ready to train! 🚀


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Device: {device}")

# EfficientNet model load karo (pretrained)
model = models.efficientnet_b0(weights='IMAGENET1K_V1')

# Last layer change karo — 12 classes ke liye
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 12)
model = model.to(device)

# Loss aur optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.5)

print("✅ Model ready!")
print(f"✅ Output classes: 12")
print("Ab training shuru karenge! 🌿")

✅ Device: cuda
✅ Model ready!
✅ Output classes: 12
Ab training shuru karenge! 🌿


In [ ]:
# Training loop
EPOCHS = 10

print("🚀 Training shuru ho rahi hai...")
print("="*55)

for epoch in range(EPOCHS):
    # ── Training phase ──
    model.train()
    train_loss = 0
    train_correct = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        _, predicted = outputs.max(1)
        train_correct += predicted.eq(labels).sum().item()

    # ── Testing phase ──
    model.eval()
    test_correct = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            test_correct += predicted.eq(labels).sum().item()

    train_acc = 100 * train_correct / len(train_data)
    test_acc  = 100 * test_correct  / len(test_data)

    scheduler.step()

    print(f"Epoch {epoch+1:2d}/{EPOCHS} | "
          f"Train Acc: {train_acc:.1f}% | "
          f"Test Acc: {test_acc:.1f}%")

print("="*55)
print("✅ Training complete!")


🚀 Training shuru ho rahi hai...
Epoch  1/10 | Train Acc: 95.1% | Test Acc: 98.2%
Epoch  2/10 | Train Acc: 98.2% | Test Acc: 99.4%
Epoch  3/10 | Train Acc: 98.7% | Test Acc: 98.3%
Epoch  4/10 | Train Acc: 99.5% | Test Acc: 99.9%
Epoch  5/10 | Train Acc: 99.7% | Test Acc: 99.7%
Epoch  6/10 | Train Acc: 99.8% | Test Acc: 99.6%
Epoch  7/10 | Train Acc: 99.8% | Test Acc: 99.8%
Epoch  8/10 | Train Acc: 99.8% | Test Acc: 99.8%
Epoch  9/10 | Train Acc: 99.9% | Test Acc: 99.7%
Epoch 10/10 | Train Acc: 100.0% | Test Acc: 99.8%
✅ Training complete!


In [ ]:
# save model
torch.save({
    'model_state_dict': model.state_dict(),
    'classes': dataset.classes,
    'accuracy': 99.8
}, 'rajasthan_crop_model.pth')

print("✅ Model saved!")
print(f"✅ Classes saved: {dataset.classes}")

# google drive
from google.colab import drive
drive.mount('/content/drive')

✅ Model saved!
✅ Classes saved: ['आलू_Early_Blight', 'आलू_Healthy', 'आलू_Late_Blight', 'टमाटर_Early_Blight', 'टमाटर_Healthy', 'टमाटर_Late_Blight', 'मक्का_Blight', 'मक्का_Healthy', 'मक्का_Rust', 'मिर्च_Bacterial_Spot', 'मिर्च_Healthy', 'सोयाबीन_Healthy']
Mounted at /content/drive


In [ ]:
import shutil

# Google Drive folder
os.makedirs('/content/drive/MyDrive/Crop_Disease_Model', exist_ok=True)

# Model copy from Drive
shutil.copy('rajasthan_crop_model.pth',
            '/content/drive/MyDrive/Crop_Disease_Model/rajasthan_crop_model.pth')

print("✅ Model saved in google drive!")
print("📁 Location: MyDrive/Crop_Disease_Model/")

✅ Model saved in google drive!
📁 Location: MyDrive/Crop_Disease_Model/


In [ ]:
!pip install gradio -q
print("✅ Gradio ready!")

✅ Gradio ready!


In [ ]:
import gradio as gr
import torch
from torchvision import transforms, models
from PIL import Image
from torch import nn

# ── Pesticide Database ────────────────────────────────────────
PESTICIDE_DB = {
    'आलू_Early_Blight': {
        'hindi': 'आलू - अगेती झुलसा',
        'pesticides': [
            {'naam': 'Mancozeb 75% WP', 'matra': '2.5 gram/litre paani', 'kimat': '₹180/kg'},
            {'naam': 'Chlorothalonil', 'matra': '2 gram/litre paani', 'kimat': '₹320/kg'},
        ],
        'spray': 'Har 7 din mein, subah ya shaam',
        'savdhani': 'Spray ke baad haath achhe se dhoyen',
        'severity': 'Moderate ⚠️'
    },
    'आलू_Late_Blight': {
        'hindi': 'आलू - पछेती झुलसा',
        'pesticides': [
            {'naam': 'Metalaxyl + Mancozeb', 'matra': '2.5 gram/litre paani', 'kimat': '₹450/kg'},
            {'naam': 'Cymoxanil 8% + Mancozeb 64%', 'matra': '3 gram/litre paani', 'kimat': '₹380/kg'},
        ],
        'spray': 'Har 5-7 din mein',
        'savdhani': 'Baarish ke baad zaroor spray karein',
        'severity': 'High 🔴'
    },
    'आलू_Healthy': {
        'hindi': 'आलू - Swasth Patti ✅',
        'pesticides': [],
        'spray': 'Koi zaroorat nahi',
        'savdhani': 'Fasal ki regular nigrani karte rahein',
        'severity': 'Healthy ✅'
    },
    'टमाटर_Early_Blight': {
        'hindi': 'टमाटर - अगेती झुलसा',
        'pesticides': [
            {'naam': 'Mancozeb 75% WP', 'matra': '2.5 gram/litre paani', 'kimat': '₹180/kg'},
            {'naam': 'Iprodione 50% WP', 'matra': '2 gram/litre paani', 'kimat': '₹520/kg'},
        ],
        'spray': 'Har 7 din mein',
        'savdhani': 'Gili pattiyaan hatayein',
        'severity': 'Moderate ⚠️'
    },
    'टमाटर_Late_Blight': {
        'hindi': 'टमाटर - पछेती झुलसा',
        'pesticides': [
            {'naam': 'Metalaxyl + Mancozeb', 'matra': '2.5 gram/litre paani', 'kimat': '₹450/kg'},
            {'naam': 'Fenamidone 10% + Mancozeb 50%', 'matra': '3 gram/litre paani', 'kimat': '₹680/kg'},
        ],
        'spray': 'Har 5 din mein',
        'savdhani': 'Infected patte turant hatayein',
        'severity': 'High 🔴'
    },
    'टमाटर_Healthy': {
        'hindi': 'टमाटर - Swasth Patti ✅',
        'pesticides': [],
        'spray': 'Koi zaroorat nahi',
        'savdhani': 'Fasal ki regular nigrani karte rahein',
        'severity': 'Healthy ✅'
    },
    'मक्का_Blight': {
        'hindi': 'मक्का - झुलसा रोग',
        'pesticides': [
            {'naam': 'Mancozeb 75% WP', 'matra': '2.5 gram/litre paani', 'kimat': '₹180/kg'},
            {'naam': 'Zineb 75% WP', 'matra': '2 gram/litre paani', 'kimat': '₹210/kg'},
        ],
        'spray': 'Har 10 din mein',
        'savdhani': 'Beej upchar zaroor karein',
        'severity': 'Moderate ⚠️'
    },
    'मक्का_Rust': {
        'hindi': 'मक्का - रतुआ रोग',
        'pesticides': [
            {'naam': 'Propiconazole 25% EC', 'matra': '1 ml/litre paani', 'kimat': '₹650/litre'},
            {'naam': 'Tebuconazole 25.9% EC', 'matra': '1 ml/litre paani', 'kimat': '₹780/litre'},
        ],
        'spray': 'Har 7-10 din mein',
        'savdhani': 'Nami kam rakhein',
        'severity': 'Moderate ⚠️'
    },
    'मक्का_Healthy': {
        'hindi': 'मक्का - Swasth Patti ✅',
        'pesticides': [],
        'spray': 'Koi zaroorat nahi',
        'savdhani': 'Fasal ki regular nigrani karte rahein',
        'severity': 'Healthy ✅'
    },
    'मिर्च_Bacterial_Spot': {
        'hindi': 'मिर्च - जीवाणु धब्बा',
        'pesticides': [
            {'naam': 'Copper Oxychloride 50% WP', 'matra': '3 gram/litre paani', 'kimat': '₹280/kg'},
            {'naam': 'Streptomycin + Tetracycline', 'matra': '1 gram/litre paani', 'kimat': '₹420/kg'},
        ],
        'spray': 'Har 7 din mein',
        'savdhani': 'Paani ka chhidkav pattiyaan par na karein',
        'severity': 'Moderate ⚠️'
    },
    'मिर्च_Healthy': {
        'hindi': 'मिर्च - Swasth Patti ✅',
        'pesticides': [],
        'spray': 'Koi zaroorat nahi',
        'savdhani': 'Fasal ki regular nigrani karte rahein',
        'severity': 'Healthy ✅'
    },
    'सोयाबीन_Healthy': {
        'hindi': 'सोयाबीन - Swasth Patti ✅',
        'pesticides': [],
        'spray': 'Koi zaroorat nahi',
        'savdhani': 'Fasal ki regular nigrani karte rahein',
        'severity': 'Healthy ✅'
    },
}

# ── Model load karo ───────────────────────────────────────────
CLASSES = ['आलू_Early_Blight','आलू_Healthy','आलू_Late_Blight',
           'टमाटर_Early_Blight','टमाटर_Healthy','टमाटर_Late_Blight',
           'मक्का_Blight','मक्का_Healthy','मक्का_Rust',
           'मिर्च_Bacterial_Spot','मिर्च_Healthy','सोयाबीन_Healthy']

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.efficientnet_b0(weights=None)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 12)
checkpoint = torch.load('rajasthan_crop_model.pth',
                         map_location=device, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model.to(device)
model.eval()

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# ── Prediction function ───────────────────────────────────────
def diagnose(image):
    img_tensor = transform(image).unsqueeze(0).to(device)
    with torch.no_grad():
        outputs = model(img_tensor)
        probs   = torch.softmax(outputs, dim=1)[0]
        conf, idx = probs.max(0)

    label = CLASSES[idx.item()]
    info  = PESTICIDE_DB[label]
    confidence = conf.item() * 100

    # ── Output banao ─────────────────────────────────────────
    result  = f"🌿 **Fasal aur Bimari:** {info['hindi']}\n\n"
    result += f"📊 **Confidence:** {confidence:.1f}%\n"
    result += f"⚠️  **Severity:** {info['severity']}\n\n"
    result += "─" * 40 + "\n\n"

    if info['pesticides']:
        result += "💊 **Suggested Pesticides (Dawai):**\n\n"
        for i, p in enumerate(info['pesticides'], 1):
            result += f"**{i}. {p['naam']}**\n"
            result += f"   • Matra : {p['matra']}\n"
            result += f"   • Kimat : {p['kimat']}\n\n"
        result += f"⏰ **Spray Schedule:** {info['spray']}\n\n"
        result += f"⚠️  **Savdhani:** {info['savdhani']}\n"
    else:
        result += "✅ **Fasal bilkul swasth hai!**\n"
        result += f"💡 **Sujhaav:** {info['savdhani']}\n"

    return result

# ── Gradio Interface ──────────────────────────────────────────
app = gr.Interface(
    fn=diagnose,
    inputs=gr.Image(type="pil", label="📸 Patti ki photo upload karein"),
    outputs=gr.Markdown(label="🌿 Result"),
    title="🌾 Rajasthan Fasal Bimari Detector",
    description="📸 Apni fasal ki patti ki photo upload karein — bimari aur dawai dono batayenge!",
    examples=[],
    theme=gr.themes.Soft()
)

app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2546abefa143cbc619.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
# Check karo kaunsi crops ka data available hai
import requests

CROPS_TO_SEARCH = [
    "wheat disease", "mustard disease", "bajra pearl millet disease",
    "cumin jeera disease", "coriander dhania disease", "garlic disease",
    "soybean disease", "chickpea chana disease", "guar cluster bean disease",
    "moth bean disease", "cotton disease", "onion disease",
    "tomato disease", "potato disease", "chilli disease",
    "okra bhindi disease", "brinjal eggplant disease", "maize disease",
    "pomegranate disease", "guava disease", "papaya disease",
    "mango disease", "lemon citrus disease", "watermelon disease",
    "fenugreek methi disease", "fennel saunf disease",
]

print("🔍 Roboflow pe available datasets:\n")
print("Manually search karo — https://universe.roboflow.com\n")
print("="*50)

available = [
    "✅ wheat disease          — ~5000 images",
    "✅ mustard disease        — ~2000 images",
    "✅ soybean disease        — ~5000 images",
    "✅ cotton disease         — ~3000 images",
    "✅ onion disease          — ~2000 images",
    "✅ tomato disease         — Already hai",
    "✅ potato disease         — Already hai",
    "✅ chilli disease         — Already hai",
    "✅ maize disease          — Already hai",
    "✅ okra bhindi disease    — ~1000 images",
    "✅ brinjal disease        — ~1000 images",
    "✅ pomegranate disease    — ~1500 images",
    "✅ guava disease          — ~1000 images",
    "✅ papaya disease         — ~1500 images",
    "✅ mango disease          — ~2000 images",
    "✅ citrus lemon disease   — ~2000 images",
    "✅ watermelon disease     — ~1000 images",
    "✅ chickpea disease       — ~1000 images",
    "❌ bajra pearl millet     — Bahut kam",
    "❌ cumin jeera            — Nahi milega",
    "❌ coriander dhania       — Nahi milega",
    "❌ garlic lahsun          — Bahut kam",
    "❌ guar cluster bean      — Nahi milega",
    "❌ moth bean              — Nahi milega",
    "❌ fenugreek methi        — Nahi milega",
]

for item in available:
    print(f"  {item}")

print(f"\n✅ Internet pe milenge  : 18 crops")
print(f"❌ Khud collect karne  : 33 crops")
print(f"📊 Total target        : 51 crops")

🔍 Roboflow pe available datasets:

Manually search karo — https://universe.roboflow.com

  ✅ wheat disease          — ~5000 images
  ✅ mustard disease        — ~2000 images
  ✅ soybean disease        — ~5000 images
  ✅ cotton disease         — ~3000 images
  ✅ onion disease          — ~2000 images
  ✅ tomato disease         — Already hai
  ✅ potato disease         — Already hai
  ✅ chilli disease         — Already hai
  ✅ maize disease          — Already hai
  ✅ okra bhindi disease    — ~1000 images
  ✅ brinjal disease        — ~1000 images
  ✅ pomegranate disease    — ~1500 images
  ✅ guava disease          — ~1000 images
  ✅ papaya disease         — ~1500 images
  ✅ mango disease          — ~2000 images
  ✅ citrus lemon disease   — ~2000 images
  ✅ watermelon disease     — ~1000 images
  ✅ chickpea disease       — ~1000 images
  ❌ bajra pearl millet     — Bahut kam
  ❌ cumin jeera            — Nahi milega
  ❌ coriander dhania       — Nahi milega
  ❌ garlic lahsun          — Bahut kam

In [ ]:
import torch
import torchvision.models as models
from torch import nn

# Same architecture jo tune use ki thi
model = models.efficientnet_b0(weights=None)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 12)

# Weights load karo
model.load_state_dict(torch.load('rajasthan_crop_model.pth', map_location='cpu'))
model.eval()

# ONNX mein convert karo
dummy_input = torch.randn(1, 3, 224, 224)
torch.onnx.export(
    model,
    dummy_input,
    "crop_disease.onnx",
    input_names=['input'],
    output_names=['output'],
    opset_version=11
)

print("✅ crop_disease.onnx ban gaya!")